# Traffic Sign Detection — Step-by-Step Walkthrough

**GTSRB traffic sign recognition with a feed-forward neural network written from scratch in NumPy.**

This notebook walks the whole project from raw `.ppm` files to a trained classifier, stopping at
every stage to show what the data looks like and what the maths is doing. There is no TensorFlow,
no PyTorch and no scikit-learn anywhere in it — forward propagation, backpropagation, Adam,
dropout and early stopping are all implemented in `src/ffnn.py`.

## What you will do

| Part | Step | What happens |
|---|---|---|
| **A. Data** | 1 | Download, extract and decode GTSRB |
| | 2 | Inspect the dataset — shapes, class balance, sample images |
| | 3 | Walk one image through the preprocessing pipeline |
| | 4 | Split into train / dev / test and encode |
| **B. The network, piece by piece** | 5 | Build the architecture and initialize parameters |
| | 6 | Forward propagation on one mini-batch |
| | 7 | The cost function |
| | 8 | Backward propagation |
| | 9 | The parameter update (Adam) |
| | 10 | One hand-rolled training step, end to end |
| **C. Training** | 11 | Train the real model with `train()` |
| | 12 | Read the learning curves |
| **D. Evaluation** | 13 | Predict on the test set |
| | 14 | Confusion matrix and per-class metrics |
| | 15 | Look at what the model got wrong |
| **E. Using the model** | 16 | Save, reload, classify a single image |
| **F. Going further** | 17 | Data augmentation |
| | 18 | Where to take this next |

## Before you start

```bash
pip install -r requirements.txt
jupyter notebook Project_Walkthrough.ipynb
```

Run the cells in order. Step 1 downloads ~300 MB the first time and takes a few minutes to
decode; every run after that loads from a cache in seconds.

---
## Step 0 — Setup

The project modules live in `src/`. We add that to the import path and pin the random seed so
this notebook reproduces the same numbers on every run.

**Note on `SAMPLE_PERCENT` and `QUICK_RUN`:** set `QUICK_RUN = True` to work with a fraction of
GTSRB and only a handful of epochs. Use it the first time through to check that everything is
wired up, then set it to `False` for a real training run.

In [ ]:
import os
import sys
import time

import numpy as np
import matplotlib.pyplot as plt

# the project modules are in src/ -- put it on the import path
PROJECT_ROOT = os.path.abspath(".")
sys.path.insert(0, os.path.join(PROJECT_ROOT, "src"))

# every relative data path in the project ("dataset/gtsrb/", "models/") is
# resolved against the working directory, so anchor it at the project root
os.chdir(PROJECT_ROOT)

np.random.seed(1)
%matplotlib inline
plt.rcParams["figure.dpi"] = 110

# ---- run configuration -------------------------------------------------------
QUICK_RUN = True          # True -> small sample, few epochs; flip to False for the real run
SAMPLE_PERCENT = 20 if QUICK_RUN else 100
NUM_EPOCHS     = 5  if QUICK_RUN else 40
HIDDEN_LAYERS  = [256, 128] if QUICK_RUN else [512, 256]

print("project root :", PROJECT_ROOT)
print("quick run    :", QUICK_RUN)
print("sample       : %d%% of GTSRB" % SAMPLE_PERCENT)
print("epochs       :", NUM_EPOCHS)
print("hidden layers:", HIDDEN_LAYERS)

In [ ]:
# ---- project modules ---------------------------------------------------------
# dataset.py : everything that turns .ppm files into (features, m) matrices
from dataset import (IMG_SIZE, NUM_CLASSES, NUM_FEATURES, DATA_ROOT, GTSRB_URLS,
                     load_dataset, train_dev_split, prep_dataset,
                     flatten_input, normalize_input, one_hot_encoding,
                     preprocess_image, label_description,
                     visualize_dataset, visualize_data_distribution)

# ffnn.py : the network itself
from ffnn import (init_layers, init_parameters, init_hyperParams,
                  forward_prop, softmax_cross_entropy_cost, backward_prop,
                  initialize_adam, update_parameters, evaluate,
                  train, predict)

# model_utils.py : activations, mini-batching, metrics, plots, persistence
from model_utils import (relu, softmax, rand_mini_batches,
                         confusion_matrix, model_metrics, metric_summary,
                         plot_confusion_matrix, visualize_training_results,
                         visualize_prediction, visualize_mislabelled_images,
                         save_model, load_model)

print("Image size   : %d x %d" % (IMG_SIZE, IMG_SIZE))
print("Input features: %d  (a flattened %dx%d grayscale crop)" % (NUM_FEATURES, IMG_SIZE, IMG_SIZE))
print("Classes       : %d" % NUM_CLASSES)

---
# Part A — The data

## Step 1 — Download, extract and decode GTSRB

The German Traffic Sign Recognition Benchmark ships as three zip archives: training images,
test images and the test ground truth. `load_dataset()` handles the whole chain and is
idempotent — each stage is skipped if its output already exists:

```
download_dataset()    -> dataset/gtsrb/*.zip          (skipped if the files are there)
decompress_dataset()  -> dataset/gtsrb/GTSRB/...      (skipped if already extracted)
retrive_dataset()     -> decode ~52,000 .ppm files    (cached to gtsrb_32.npz)
sample_dataset()      -> shuffle and take SAMPLE_PERCENT
```

The decode step is the slow one — a couple of minutes — so its output is cached as a
compressed `.npz`. Later runs load in seconds.

**If the download is blocked on your network:** fetch the three archives listed below by hand,
drop them in `dataset/gtsrb/`, and rerun this cell.

In [ ]:
print("Archives GTSRB is fetched from:\n")
for name, url in GTSRB_URLS.items():
    print("  %-38s %s" % (name, url))
print("\nThey are downloaded into: %s" % DATA_ROOT)

In [ ]:
tic = time.time()

# The first call downloads (~300 MB), extracts and decodes; later calls hit the .npz cache.
train_x_orig, train_y_orig, test_x_orig, test_y_orig = load_dataset(size_in_per=SAMPLE_PERCENT)

print("\nLoaded in %.1f s\n" % (time.time() - tic))
print("Set            Images                     Labels")
print("=" * 62)
print("Training       %-26s %s" % (train_x_orig.shape, train_y_orig.shape))
print("Test           %-26s %s" % (test_x_orig.shape, test_y_orig.shape))
print("=" * 62)
print("\ndtype: images %s (0-255), labels %s (0-42)"
      % (train_x_orig.dtype, train_y_orig.dtype))

## Step 2 — Inspect the dataset

Two things are worth looking at before touching the model.

**Class balance.** GTSRB is genuinely imbalanced — the commonest class has roughly ten times
the examples of the rarest. That matters later: overall accuracy will look better than the
model deserves, because getting the big classes right carries the average. This is why Step 14
reports **per-class** precision and recall, not just accuracy.

**What the images actually look like.** These are photographs of real signs taken from a moving
car — motion blur, shadow, glare, occlusion, signs at an angle. They are nothing like the clean
centred glyphs of MNIST.

In [ ]:
labels, counts = np.unique(train_y_orig, return_counts=True)
label_desc = label_description()

print("Classes present : %d of %d" % (len(labels), NUM_CLASSES))
print("Largest class   : %-28s %d examples" % (label_desc[int(labels[counts.argmax()])], counts.max()))
print("Smallest class  : %-28s %d examples" % (label_desc[int(labels[counts.argmin()])], counts.min()))
print("Imbalance ratio  : %.1fx" % (counts.max() / counts.min()))

visualize_data_distribution(train_y_orig, dataset_type="training")

In [ ]:
# 10 random preprocessed training images with their labels
visualize_dataset(train_x_orig, train_y_orig, dataset="gtsrb", dataset_type="training")

In [ ]:
# The full class list -- useful to keep handy while reading the confusion matrix later
print("id  name")
print("-" * 42)
for class_id, name in label_desc.items():
    print("%2d  %s" % (class_id, name))

## Step 3 — What preprocessing does to one image

Every image passes through `dataset.preprocess_image()`. Five steps, each one earning its place:

1. **Crop to the annotated ROI.** GTSRB gives a bounding box per image. The raw file carries
   about 10% margin of pure background around the sign — noise a fully-connected net would
   happily memorise.
2. **Convert to grayscale.** Working in RGB would triple the input weights. Sign *shape* and
   *pictogram* carry most of the signal; the red/blue distinction is largely redundant with them.
3. **Equalize the histogram.** The highest-value step for GTSRB. Images are shot in tunnels,
   direct sun and dusk. Equalizing makes a night shot and a noon shot of the same sign look
   comparable to the network.
4. **Resize to 32×32.** The network needs a fixed input size.
5. **Flatten and scale.** `(32, 32) → (1024, 1)`, then divide by 255 so every feature sits in
   `[0, 1]`. Unscaled inputs make the first layer's gradients enormous and training unstable.

The cell below runs the pipeline on a raw file so you can see each stage. If the extracted
`.ppm` files are not on disk (you loaded from the cache), it falls back to showing the cached
result on its own.

In [ ]:
from PIL import Image, ImageOps

raw_dir = os.path.join(DATA_ROOT, "GTSRB", "Final_Training", "Images", "00014")  # class 14 = Stop
raw_files = sorted(f for f in os.listdir(raw_dir)) if os.path.exists(raw_dir) else []
raw_ppm = [f for f in raw_files if f.endswith(".ppm")]

if raw_ppm:
    import csv
    with open(os.path.join(raw_dir, "GT-00014.csv"), newline="") as fh:
        row = next(csv.DictReader(fh, delimiter=";"))
    roi = (int(row["Roi.X1"]), int(row["Roi.Y1"]), int(row["Roi.X2"]), int(row["Roi.Y2"]))
    original = Image.open(os.path.join(raw_dir, row["Filename"]))

    # reproduce preprocess_image() one stage at a time
    stage_cropped   = original.crop(roi)
    stage_gray      = stage_cropped.convert("L")
    stage_equalized = ImageOps.equalize(stage_gray)
    stage_resized   = stage_equalized.resize((IMG_SIZE, IMG_SIZE), Image.BILINEAR)

    stages = [(original, "1. raw %dx%d" % original.size),
              (stage_cropped, "2. cropped to ROI"),
              (stage_gray, "3. grayscale"),
              (stage_equalized, "4. equalized"),
              (stage_resized, "5. resized %dx%d" % (IMG_SIZE, IMG_SIZE))]

    fig, axes = plt.subplots(1, 5, figsize=(16, 4))
    for ax, (img, title) in zip(axes, stages):
        ax.imshow(img, cmap=None if img.mode == "RGB" else "gray")
        ax.set_title(title, fontsize=10)
        ax.axis("off")
    fig.suptitle("preprocess_image() stage by stage", fontsize=13)
    plt.show()
else:
    print("Raw .ppm files not on disk (loaded from the .npz cache) -- showing a cached example.")
    plt.figure(figsize=(3, 3))
    plt.imshow(train_x_orig[0], cmap="gray")
    plt.title("cached %dx%d crop, label %d" % (IMG_SIZE, IMG_SIZE, train_y_orig[0, 0]))
    plt.axis("off")
    plt.show()

In [ ]:
# The last two steps, on the arrays this time: flatten, then normalize.
one_image = train_x_orig[0]
print("as an image          : %-14s  range %d..%d" % (one_image.shape, one_image.min(), one_image.max()))

flat = flatten_input(train_x_orig[:1])
print("after flatten_input  : %-14s  range %d..%d" % (flat.shape, flat.min(), flat.max()))

norm = normalize_input(flat)
print("after normalize_input: %-14s  range %.2f..%.2f" % (norm.shape, norm.min(), norm.max()))

print("\nNote the orientation: features down the rows, examples across the columns.")
print("The whole project is column-major, so a mini-batch is a contiguous slice of columns.")

## Step 4 — Split and encode

Three sets, each with a distinct job:

- **Training** — the model fits its weights on this.
- **Development (dev)** — held out from training, checked once per epoch. It drives early
  stopping and tells you when you have started overfitting. Carved out of the training set, so
  the test set stays untouched.
- **Test** — touched exactly once, at the very end, in Step 13. Every time you look at the test
  set and then change something, you leak a little information into your model.

Labels become one-hot columns: class 14 turns into a height-43 column of zeros with a single 1
at row 14. That is the shape softmax cross-entropy expects.

In [ ]:
train_x, train_y, dev_x, dev_y = train_dev_split(train_x_orig, train_y_orig, dev_fraction=0.1)

print("Set          Images                     Labels")
print("=" * 60)
print("Training     %-26s %s" % (train_x.shape, train_y.shape))
print("Dev          %-26s %s" % (dev_x.shape, dev_y.shape))
print("Test         %-26s %s" % (test_x_orig.shape, test_y_orig.shape))
print("=" * 60)

In [ ]:
# prep_dataset = flatten_input -> normalize_input -> one_hot_encoding
train_x_norm, train_y_enc = prep_dataset(train_x, train_y)
dev_x_norm,   dev_y_enc   = prep_dataset(dev_x, dev_y)
test_x_norm,  test_y_enc  = prep_dataset(test_x_orig, test_y_orig)

print("Set          Before               After")
print("=" * 56)
print("Train X      %-20s %s" % (train_x.shape, train_x_norm.shape))
print("Train Y      %-20s %s" % (train_y.shape, train_y_enc.shape))
print("Dev X        %-20s %s" % (dev_x.shape, dev_x_norm.shape))
print("Test X       %-20s %s" % (test_x_orig.shape, test_x_norm.shape))
print("=" * 56)

# what one-hot encoding did to a single label
i = 0
print("\nExample %d: label %d (%s)" % (i, train_y[i, 0], label_desc[int(train_y[i, 0])]))
print("one-hot column shape %s, 1 at row %d, %d zeros elsewhere"
      % (train_y_enc[:, i].shape, train_y_enc[:, i].argmax(), (train_y_enc[:, i] == 0).sum()))

---
# Part B — The network, piece by piece

Steps 5 to 10 open up `train()` and run each of its internal stages by hand on a single
mini-batch. Nothing here is needed to train the model — Step 11 does that in one call — but it
is where you can actually see the shapes flowing through and check that the maths matches what
you expect.

## Step 5 — Architecture and parameter initialization

`init_layers` just builds the list of layer widths. `init_parameters` creates a weight matrix
`W[l]` of shape `(n[l], n[l-1])` and a bias vector `b[l]` of shape `(n[l], 1)` for each layer.

**Why He initialization?** Weights drawn from `N(0, 1)` scaled by `√(2/n_prev)`. Too small and
the signal dies out as it passes through layers; too large and it explodes. The `√(2/n_prev)`
factor is the scale that keeps the variance of the activations roughly constant through a ReLU
network. The alternative, `random` (a flat ×0.01), is kept for comparison — try it in Step 11
and watch training start noticeably slower.

Biases start at zero. They have no symmetry to break — the weights already do that.

In [ ]:
layers_dim = init_layers(NUM_FEATURES, NUM_CLASSES, hidden_layers=HIDDEN_LAYERS)
print("layers_dim =", layers_dim)
print("  input  : %d features (the flattened %dx%d crop)" % (layers_dim[0], IMG_SIZE, IMG_SIZE))
for i, n in enumerate(layers_dim[1:-1], start=1):
    print("  hidden %d: %d units, ReLU" % (i, n))
print("  output : %d units, softmax" % layers_dim[-1])

In [ ]:
parameters = init_parameters(layers_dim, initialization="he")

total = 0
print("Layer   Weight            Bias         Parameters")
print("=" * 54)
for l in range(1, len(layers_dim)):
    W, b = parameters["W" + str(l)], parameters["b" + str(l)]
    n = W.size + b.size
    total += n
    print("%d       %-17s %-12s %s" % (l, W.shape, b.shape, format(n, ",")))
print("=" * 54)
print("Total trainable parameters: %s" % format(total, ","))

print("\nW1 statistics -- mean %.5f, std %.5f (He target std = %.5f)"
      % (parameters["W1"].mean(), parameters["W1"].std(), np.sqrt(2 / layers_dim[0])))

## Step 6 — Forward propagation

For each hidden layer `l`:

$$Z^{[l]} = W^{[l]} A^{[l-1]} + b^{[l]} \qquad A^{[l]} = \text{ReLU}(Z^{[l]})$$

and at the output layer, softmax instead of ReLU, turning the final scores into a probability
distribution over the 43 classes (each column sums to 1).

Two implementation details worth knowing:

- **`caches`.** Each layer stashes `(A_prev, W, b)` and `Z` on the way forward. Backpropagation
  needs exactly those values, and recomputing them would double the work.
- **Inverted dropout.** With `regularizer="dropout"`, each hidden activation is multiplied by a
  random 0/1 mask and then *divided by* `keep_prob`. The division is what makes it "inverted":
  it keeps the expected value of the activation unchanged, so prediction time needs no
  correction at all — you simply do not apply dropout. `forward_prop` returns the masks so
  backpropagation can reuse the same ones.

Dropout is applied to hidden layers only, never to the output.

In [ ]:
# one mini-batch to trace through
batch_X = train_x_norm[:, :64]
batch_Y = train_y_enc[:, :64]
keep_probs = [0.8] * len(HIDDEN_LAYERS)

AL, caches, dropout_masks = forward_prop(batch_X, parameters,
                                         keep_probs=keep_probs, regularizer="dropout")

print("input  X : %s" % (batch_X.shape,))
print("output AL: %s   (43 class probabilities x 64 examples)" % (AL.shape,))
print("caches   : %d entries, one per layer" % len(caches))
print("masks    : %d entries, one per hidden layer" % len(dropout_masks))

print("\nEvery column of AL is a probability distribution:")
print("  column sums -> min %.6f, max %.6f" % (AL.sum(axis=0).min(), AL.sum(axis=0).max()))
print("\nUntrained network, so the predictions are near-uniform (1/43 = %.4f):" % (1 / NUM_CLASSES))
print("  highest probability in column 0: %.4f" % AL[:, 0].max())

In [ ]:
# what a dropout mask looks like
mask = dropout_masks[0]
print("mask for hidden layer 1: %s, dtype %s" % (mask.shape, mask.dtype))
print("fraction kept: %.3f  (keep_prob was %.2f)" % (mask.mean(), keep_probs[0]))

# and the same batch WITHOUT dropout -- this is what prediction does
AL_nodrop, _, _ = forward_prop(batch_X, parameters)
print("\nWithout dropout, AL shape %s, %d masks returned" % (AL_nodrop.shape, len(_)))

## Step 7 — The cost function

Softmax cross-entropy, averaged over the batch:

$$J = -\frac{1}{m}\sum_{i=1}^{m}\sum_{k=1}^{43} y_k^{(i)} \log \hat{y}_k^{(i)}$$

Only the true class contributes to each example's term (every other `y` is zero), so this is
really "the negative log of the probability the model assigned to the right answer".

**Why `from_logits=True`?** Computing `log(softmax(z))` in two steps means taking the log of a
number that can underflow to exactly 0, and `log(0)` is `-inf`. Passing the raw scores `z`
lets the function use the numerically stable identity

$$\log \text{softmax}(z) = z - \max(z) - \log \sum e^{z - \max(z)}$$

which never underflows. This is why the caches carry `Z` and not just `A`.

With `regularizer="l2"` an extra $\frac{\lambda}{2m}\sum \|W\|^2$ term is added, penalising
large weights. Note that dropout and L2 are alternatives here, not combined.

In [ ]:
cost_plain = softmax_cross_entropy_cost(AL, batch_Y, caches, from_logits=True)
cost_l2    = softmax_cross_entropy_cost(AL, batch_Y, caches, lambd=0.7,
                                        regularizer="l2", from_logits=True)

print("cost (no regularization) : %.6f" % cost_plain)
print("cost (with L2, lambda=0.7): %.6f" % cost_l2)
print("L2 penalty contributed    : %.6f" % (cost_l2 - cost_plain))

print("\nSanity check: an untrained net guesses uniformly, so the cost should")
print("start near -log(1/43) = %.4f" % (-np.log(1 / NUM_CLASSES)))

## Step 8 — Backward propagation

The gradient of softmax cross-entropy with respect to the output layer's pre-activation is
famously simple:

$$dZ^{[L]} = \hat{Y} - Y$$

The softmax derivative and the cross-entropy derivative cancel almost entirely — which is why
`backward_activation` for `"softmax"` just passes `dA` straight through as `dZ`.

From there the chain rule runs backwards through each layer:

$$dW^{[l]} = \frac{1}{m} dZ^{[l]} A^{[l-1]T} \qquad
db^{[l]} = \frac{1}{m}\sum dZ^{[l]} \qquad
dA^{[l-1]} = W^{[l]T} dZ^{[l]}$$

and the ReLU gradient is just "zero out wherever `Z` was negative" — `relu_grad` does this in
place on a copy of `dA`.

Dropout is applied here too: the *same* masks from the forward pass, so a unit that was dropped
receives no gradient.

**Shape check.** Every gradient must have the same shape as the parameter it belongs to. The
assertions inside `backward_grad` enforce this — a shape mismatch here is the single most
common bug in a from-scratch implementation, and catching it at the source beats debugging a
silent broadcast three layers later.

In [ ]:
grads = backward_prop(AL, batch_Y, caches,
                      dropout_masks=dropout_masks, keep_probs=keep_probs,
                      regularizer="dropout")

print("Parameter   shape             Gradient    shape             match")
print("=" * 68)
for l in range(1, len(layers_dim)):
    for p in ("W", "b"):
        param, grad = parameters[p + str(l)], grads["d" + p + str(l)]
        print("%-11s %-17s %-11s %-17s %s"
              % (p + str(l), param.shape, "d" + p + str(l), grad.shape,
                 "yes" if param.shape == grad.shape else "NO"))
print("=" * 68)

print("\nGradient magnitudes (a healthy net has non-zero, non-exploding gradients):")
for l in range(1, len(layers_dim)):
    g = grads["dW" + str(l)]
    print("  dW%d  mean |g| = %.3e   max |g| = %.3e" % (l, np.abs(g).mean(), np.abs(g).max()))

## Step 9 — The parameter update (Adam)

Plain gradient descent is `W := W - α·dW`. Adam adds two running averages per parameter:

- **`v`** — an exponentially weighted average of the gradient (momentum). Smooths out noisy
  mini-batch gradients and carries the update through flat regions.
- **`s`** — an exponentially weighted average of the *squared* gradient. Dividing by `√s`
  gives every parameter its own effective learning rate: parameters with consistently large
  gradients get smaller steps, rarely-updated ones get larger steps.

$$W := W - \alpha \frac{v^{corrected}}{\sqrt{s^{corrected}} + \epsilon}$$

**Bias correction.** `v` and `s` start at zero, so early estimates are biased towards zero.
Dividing by `1 - β^t` (where `t` counts updates, not epochs) corrects this. That is why `t` is
incremented per mini-batch inside `train()`, and why it must never be reset.

In [ ]:
v, s = initialize_adam(parameters)
print("Adam moment buffers: %d entries each, matching every W and b" % len(v))

W1_before = parameters["W1"].copy()
t = 1  # update counter -- drives bias correction, increments once per mini-batch

parameters, v, s = update_parameters(parameters, grads, learning_rate=0.001,
                                     optimizer="adam", beta1=0.9, beta2=0.999,
                                     epsilon=1e-8, v=v, s=s, t=t)

delta = parameters["W1"] - W1_before
print("\nAfter one Adam step:")
print("  mean |ΔW1| = %.3e" % np.abs(delta).mean())
print("  max  |ΔW1| = %.3e" % np.abs(delta).max())
print("\nWith bias correction at t=1, the step size is close to the learning rate itself,")
print("which is exactly what Adam is designed to do -- the update is scale-free.")

## Step 10 — One full training step, by hand

Putting Steps 6 to 9 together. This is the inner loop of `train()`, minus the bookkeeping.
Running it repeatedly on the *same* mini-batch should drive the cost down fast — the network is
memorising 64 examples, which is not learning, but it does prove the gradients point downhill.

**If this cost does not fall, nothing else in the project will work.** It is the cheapest
possible check that forward, backward and update all agree with each other.

In [ ]:
# fresh parameters, then overfit a single mini-batch on purpose
demo_params = init_parameters(layers_dim, initialization="he")
demo_v, demo_s = initialize_adam(demo_params)

print("Overfitting one 64-example mini-batch (a gradient sanity check):\n")
print("step   cost      batch accuracy")
print("-" * 34)

for step in range(1, 31):
    # 1. forward (no dropout -- we WANT to overfit here)
    AL_d, caches_d, _ = forward_prop(batch_X, demo_params)
    # 2. cost
    cost_d = softmax_cross_entropy_cost(AL_d, batch_Y, caches_d, from_logits=True)
    # 3. backward
    grads_d = backward_prop(AL_d, batch_Y, caches_d)
    # 4. update
    demo_params, demo_v, demo_s = update_parameters(
        demo_params, grads_d, learning_rate=0.005, optimizer="adam",
        beta1=0.9, beta2=0.999, epsilon=1e-8, v=demo_v, s=demo_s, t=step)

    if step % 5 == 0 or step == 1:
        acc, _ = evaluate(batch_X, batch_Y, demo_params)
        print("%4d   %7.4f   %.4f" % (step, cost_d, acc))

print("\nCost falling and accuracy climbing towards 1.0 -> the gradients are correct.")

---
# Part C — Training

## Step 11 — Train the model

`train()` runs everything Part B did by hand, plus the machinery that makes it a real training
run:

- **Mini-batching.** `rand_mini_batches()` reshuffles the training set every epoch (with a
  different seed each time) and cuts it into batches. Reshuffling matters: fixed batch
  composition lets the model latch onto batch-level artefacts.
- **Validation.** Dev accuracy and loss computed once per epoch. Watch the gap between train
  and dev accuracy — that gap *is* overfitting.
- **Early stopping.** With `patience=N`, the parameters are checkpointed whenever dev accuracy
  improves. If N epochs pass without improvement, training halts and the *best* parameters are
  restored, not the last ones. This is what stops a long run from quietly degrading.
- **Learning-rate decay.** Optional (`step_decay`). Large steps early to cover ground, smaller
  steps later to settle into a minimum instead of bouncing around it.

**Choosing an optimizer:** `adam` is the default and almost always the right call. `mgd` is
plain mini-batch gradient descent, `bgd` uses the whole training set as one batch (slow,
smooth), `sgd` uses one example at a time (fast per step, very noisy).

**Choosing a regularizer:** `dropout` or `l2`, not both. Start with dropout at `keep_prob=0.8`.

In [ ]:
hyper_params = init_hyperParams(
    alpha=0.001,             # learning rate
    num_epoch=NUM_EPOCHS,
    minibatch_size=128,
    lambd=0,                 # L2 strength -- unused, we are using dropout
    keep_probs=[0.8] * len(HIDDEN_LAYERS),
    beta1=0.9, beta2=0.999, epsilon=1e-8,   # Adam defaults
)

for k, val in hyper_params.items():
    print("%-16s %s" % (k, val))

In [ ]:
tic = time.time()

history = train(
    training_data=(train_x_norm, train_y_enc),
    validation_data=(dev_x_norm, dev_y_enc),
    layers_dim=layers_dim,
    hyperParams=hyper_params,
    initialization="he",
    optimizer="adam",
    regularizer="dropout",
    verbose=3,          # 0 silent | 1 epoch | 2 progress bar | 3 progress bar + metrics
    patience=5,         # early stopping; None or 0 disables it
    step_decay=None,    # e.g. 10 to decay the learning rate every 10 epochs
)

trained_params = history["parameters"]
print("\nWall clock: %.1f s" % (time.time() - tic))

## Step 12 — Read the learning curves

Two plots, and what each shape means:

| What you see | What it means | What to do |
|---|---|---|
| Both losses falling, close together | Healthy. Still learning. | Train longer, or grow the network |
| Train loss falls, dev loss flattens then rises | Overfitting | More dropout, more data, smaller network |
| Both losses flat and high | Underfitting, or the learning rate is wrong | Bigger network, or tune `alpha` |
| Loss jumping around wildly | Learning rate too high | Lower `alpha`, or enable `step_decay` |
| Loss becomes `nan` | Numerical blow-up | Lower `alpha`; check inputs are normalized |

With `QUICK_RUN = True` the curves will still be climbing steeply — five epochs on a fifth of
the data is nowhere near converged. That is expected.

In [ ]:
visualize_training_results(history["accuracy"], history["val_accuracy"],
                           history["loss"], history["val_loss"])

print("Epoch   train acc   dev acc   train loss   dev loss")
print("=" * 52)
for e in range(len(history["accuracy"])):
    print("%5d   %9.4f   %7.4f   %10.4f   %8.4f"
          % (e + 1, history["accuracy"][e], history["val_accuracy"][e],
             history["loss"][e], history["val_loss"][e]))
print("=" * 52)

gap = history["accuracy"][-1] - history["val_accuracy"][-1]
print("\nFinal train-dev accuracy gap: %.4f" % gap)
print("A widening gap is the signature of overfitting." if gap > 0.05
      else "The gap is small -- the model is not overfitting yet.")

---
# Part D — Evaluation

## Step 13 — Predict on the test set

This is the first and only time the test set is used. `predict()` runs a forward pass **without
dropout** (that is what inverted dropout bought us — no rescaling needed) and reads off two
things per example:

- **First prediction** — `argmax` of the softmax column, and the probability there.
- **Second guess** — `argmax` again after zeroing out the winner.

The second guess is genuinely useful on GTSRB, where several classes are near-identical at
32×32: the speed limits differ only in their digits, and "Dangerous curve left" versus
"Dangerous curve right" is a mirror of the same triangle. A model that is wrong first but right
second is confusing a specific pair, not failing to see the sign at all.

In [ ]:
prediction = predict(test_x_norm, trained_params, second_guess=True)

first_lbl, first_prob = prediction["First Prediction"]
sec_lbl,   sec_prob   = prediction["Second Prediction"]

true_lbl = test_y_orig.T   # (1, m) to match the prediction arrays

top1 = np.mean(first_lbl == true_lbl)
top2 = np.mean((first_lbl == true_lbl) | (sec_lbl == true_lbl))

print("Test examples : %d" % true_lbl.shape[1])
print("Top-1 accuracy: %.4f" % top1)
print("Top-2 accuracy: %.4f" % top2)
print("Rescued by the second guess: %.4f" % (top2 - top1))
print("\nMean confidence when right: %.4f" % first_prob[first_lbl == true_lbl].mean())
print("Mean confidence when wrong: %.4f" % first_prob[first_lbl != true_lbl].mean())
print("\nA model that is much less confident when wrong is well calibrated -- its")
print("confidence score is worth thresholding on in a real deployment.")

## Step 14 — Confusion matrix and per-class metrics

Accuracy alone is misleading on an imbalanced dataset — a model that ignores every rare class
entirely can still post a respectable number. The per-class table is where the truth is:

- **Precision** — of everything predicted as class *k*, what fraction really was? Low precision
  means the model over-uses that label.
- **Recall** — of everything that really was class *k*, what fraction did it catch? Low recall
  means it misses that sign.
- **F1** — their harmonic mean; low if *either* is low.
- **Macro average** — the unweighted mean across all 43 classes. Unlike accuracy, it gives a
  rare class exactly as much weight as a common one, so it is the honest headline number here.

In the 43×43 heat map, the diagonal is correct predictions. Bright off-diagonal cells are the
confusions worth investigating — look for clusters among the speed limits (classes 0–8) and the
triangular warning signs (18–31).

In [ ]:
cm = confusion_matrix(test_y_orig, prediction, num_classes=NUM_CLASSES)
print("confusion matrix: %s  (rows = true class, columns = predicted)" % (cm.shape,))
print("correct predictions on the diagonal: %d of %d" % (np.trace(cm), cm.sum()))

plot_confusion_matrix(cm, dataset_type="test")

In [ ]:
metrics, macro_metrics, acc = model_metrics(cm)
metric_summary(metrics, macro_metrics, acc)

In [ ]:
# Which classes does the model handle worst, and is it a data-volume problem?
f1 = np.array(metrics["F1-Score"])
support = cm.sum(axis=1)
order = np.argsort(f1)

print("Ten weakest classes")
print("=" * 68)
print("id  name                          F1       precision  recall  support")
print("-" * 68)
for k in order[:10]:
    print("%2d  %-28s %.4f   %.4f     %.4f  %d"
          % (k, label_desc[int(k)], f1[k],
             metrics["Precision"][k], metrics["Recall"][k], support[k]))
print("=" * 68)

seen = support > 0
# np.corrcoef is undefined if either series is constant, so guard it
if seen.sum() > 2 and support[seen].std() > 0 and f1[seen].std() > 0:
    corr = np.corrcoef(support[seen], f1[seen])[0, 1]
    print("\nCorrelation between training support and F1: %+.3f" % corr)
    print("A strong positive correlation says the weak classes are weak because they")
    print("are rare -- which points at class weighting or oversampling, not architecture.")
else:
    print("\nNot enough variation across classes to correlate support against F1.")

In [ ]:
# The most confused PAIRS -- usually more actionable than any single class
cm_off = cm.copy()
np.fill_diagonal(cm_off, 0)

print("Most frequent confusions")
print("=" * 78)
flat_order = np.argsort(cm_off, axis=None)[::-1][:10]
for idx in flat_order:
    i, j = np.unravel_index(idx, cm_off.shape)
    if cm_off[i, j] == 0:
        break
    print("%3d x   %-28s  ->  %s" % (cm_off[i, j], label_desc[int(i)], label_desc[int(j)]))
print("=" * 78)

## Step 15 — Look at what it got wrong

Metrics tell you *how much* is wrong; the images tell you *why*. Scan the mislabelled grid for
patterns — heavily blurred crops, signs at a steep angle, near-identical pictograms, or images
where even you cannot tell what the sign is at 32×32. Each pattern points at a different fix
(more augmentation, a higher input resolution, colour channels, a convolutional model).

In [ ]:
visualize_prediction(test_x_orig, test_y_orig.T, prediction, dataset_type="test")

In [ ]:
visualize_mislabelled_images(test_x_orig, test_y_orig.T, prediction, dataset_type="test")

---
# Part E — Using the model

## Step 16 — Save, reload, classify a single image

The model is pickled as a dictionary — the parameters plus enough metadata (`layers_dim`,
`img_size`, `num_classes`) for anything loading it to know what it expects. `predict.py` and
`gui.py` both read exactly this format, so a model saved here works with both immediately.

In [ ]:
model = {
    "parameters":  trained_params,
    "layers_dim":  layers_dim,
    "img_size":    IMG_SIZE,
    "num_classes": NUM_CLASSES,
    "hyper_params": hyper_params,
    "optimizer":   "adam",
    "regularizer": "dropout",
    "history": {k: history[k] for k in ("accuracy", "loss", "val_accuracy", "val_loss")},
}

os.makedirs("models", exist_ok=True)
MODEL_PATH = "models/gtsrb_mlp_notebook"
save_model(MODEL_PATH, model)
print("saved -> %s (%.1f MB)" % (MODEL_PATH, os.path.getsize(MODEL_PATH) / 1e6))

reloaded = load_model(MODEL_PATH)
print("reloaded keys:", sorted(reloaded.keys()))
print("architecture :", reloaded["layers_dim"])

In [ ]:
def classify_one(image_array, parameters, true_label=None):
    # Classify a single preprocessed (IMG_SIZE, IMG_SIZE) array and plot the result.
    column = image_array.reshape(-1, 1) / 255.        # flatten and normalize, same as training
    pred = predict(column, parameters, second_guess=True)

    lbl1, p1 = pred["First Prediction"]
    lbl2, p2 = pred["Second Prediction"]
    lbl1, p1, lbl2, p2 = int(lbl1), float(p1), int(lbl2), float(p2)

    fig, ax = plt.subplots(1, 2, figsize=(10, 4))
    ax[0].imshow(image_array, cmap="gray")
    ax[0].axis("off")
    title = "true: %s" % label_desc[int(true_label)] if true_label is not None else "input"
    ax[0].set_title(title, fontsize=10)

    ax[1].barh([1, 0], [p1, p2])
    ax[1].set_yticks([1, 0])
    ax[1].set_yticklabels([label_desc[lbl1], label_desc[lbl2]], fontsize=9)
    ax[1].set_xlim(0, 1)
    ax[1].set_xlabel("probability")
    ax[1].set_title("1st: %.1f%%   2nd: %.1f%%" % (p1 * 100, p2 * 100), fontsize=10)
    plt.tight_layout()
    plt.show()

    return lbl1, p1


# three random test images through the single-image path
for i in np.random.randint(0, test_x_orig.shape[0], 3):
    classify_one(test_x_orig[i], reloaded["parameters"], true_label=test_y_orig[i, 0])

To classify your **own** photo of a sign, pass it through `preprocess_image()` first — the model
only ever sees images that went through that exact pipeline, so anything else will be
misread:

```python
from PIL import Image
from dataset import preprocess_image

with Image.open("my_sign.jpg") as img:
    array = preprocess_image(img.copy())   # add roi=(x1,y1,x2,y2) to crop first
classify_one(array, reloaded["parameters"])
```

Crop tightly to the sign yourself. GTSRB images are tight crops, and a wide street scene fed in
whole will not resemble anything in the training set.

The same thing from the command line and with a GUI:

```bash
python predict.py my_sign.jpg --model models/gtsrb_mlp_notebook
python gui.py --model models/gtsrb_mlp_notebook
```

---
# Part F — Going further

## Step 17 — Data augmentation

`src/data_augmentation.py` synthesises extra training examples with five transforms:
crop-and-pad, rotate, shift, blur and zoom. Each pass produces five variants per image, so one
pass over the training set roughly sixes it.

**The transforms are deliberately milder than in the digit version of this project** — ±15°
rotation instead of ±60°, ±3 px shift instead of ±7. A digit rotated 40° is still that digit; a
traffic sign rotated 40° is a photograph you will never encounter, and training on it wastes
capacity.

**Horizontal flip is excluded entirely, and this is not a tuning choice.** Flipping "Turn left
ahead" produces a perfectly good image of "Turn right ahead" that still carries the *left*
label. It manufactures wrong training data. The function still exists in the module with a
warning on it; `data_generator()` never calls it.

In [ ]:
from data_augmentation import (rotate_images, shift_images, blur_images,
                               zoom_images, crop_and_pad_images,
                               MAX_ROTATION, MAX_SHIFT)

print("MAX_ROTATION = %d degrees   (the MNIST version used 60)" % MAX_ROTATION)
print("MAX_SHIFT    = %d pixels    (the MNIST version used 7 on a 28x28 image)" % MAX_SHIFT)

sample = train_x[:1]
sample_lbl = train_y[:1]

variants = [(sample[0], "original"),
            (rotate_images(sample, sample_lbl)[0][0], "rotate"),
            (shift_images(sample, sample_lbl)[0][0], "shift"),
            (blur_images(sample, sample_lbl)[0][0], "blur"),
            (zoom_images(sample, sample_lbl)[0][0], "zoom"),
            (crop_and_pad_images(sample, sample_lbl)[0][0], "crop and pad")]

fig, axes = plt.subplots(1, 6, figsize=(15, 3))
for ax, (img, title) in zip(axes, variants):
    ax.imshow(img, cmap="gray")
    ax.set_title(title, fontsize=10)
    ax.axis("off")
fig.suptitle("Augmentation transforms -- %s" % label_desc[int(sample_lbl[0, 0])], fontsize=12)
plt.show()

Generating the full augmented set is slow and memory-hungry, so it runs offline as a script
rather than in this notebook:

```bash
python augment.py --aug-count 1        # writes dataset/augmented_data/batch_1
python train.py --use-augmented --epochs 30
```

`augment.py` augments only the *training* split — the dev set has to stay untouched, or early
stopping is validating against data it has effectively seen.

## Step 18 — Where to take this next

**Things to try inside this architecture**

- Set `QUICK_RUN = False` and train properly. Everything above was on a fifth of the data.
- Compare `initialization="he"` against `"random"` over the first few epochs.
- Compare `regularizer="dropout"` against `"l2"` at a few values of `lambd`.
- Add `step_decay=10` and see whether the late epochs settle more cleanly.
- Vary `HIDDEN_LAYERS`: wider, deeper, and both, and watch the train-dev gap respond.
- Address the class imbalance — oversample the rare classes and re-read the macro F1.

**The ceiling, honestly stated**

A fully-connected network on flattened pixels has no translation invariance and no notion of
locality — it learns every pixel position independently, so a sign shifted three pixels right is
a substantially different input to it. On GTSRB it lands well below what a small convolutional
network reaches on the same data, and the gap is the point of the exercise: this project is
about implementing the mechanics, not about topping the benchmark.

The natural next steps, in increasing order of effort:

1. **Keep the colour channels.** Red versus blue separates prohibitory from mandatory signs, and
   grayscale throws that away. `(32, 32, 3)` → 3072 features.
2. **Convolutional layers.** The right inductive bias for images. `src/dataset.py` will feed a
   CNN unchanged — just skip `flatten_input`.
3. **Detection, not just recognition.** This model classifies an image that already contains one
   cropped sign. Finding signs in a street scene is a different problem needing a detector.